# Math Tutor Backend - Colab Edition

This notebook implements the **4-Stage Intelligence Pipeline** for the Math Tutor.

**Colab Specifics:**
1.  Installs `tesseract-ocr` using `apt-get`.
2.  Uses `ngrok` to expose the Flask server to the public internet so your local frontend can reach it.

## Instructions
1.  Add your **Google Gemini API Key**.
2.  Add your **Ngrok Authtoken** (Get it from [dashboard.ngrok.com](https://dashboard.ngrok.com)).
3.  **Run the Setup Cell** (installs libraries).
4.  **Run the Main Application Cell** (starts the server).
5.  Copy the **public URL** printed at the end (e.g., `https://xxxx.ngrok-free.app`) and use it in your frontend.

In [ ]:
# --- CELL 1: SETUP ---
# Install system dependencies for OCR
!sudo apt-get install tesseract-ocr
!pip install pytesseract

# Install Python packages
!pip install flask flask-cors google-generativeai pillow pyngrok

In [ ]:
# --- CELL 2: MAIN APPLICATION ---
import os
import json
import google.generativeai as genai
from flask import Flask, request, jsonify
from flask_cors import CORS
from PIL import Image
import pytesseract
import io
from pyngrok import ngrok

# ==========================================
# CONFIGURATION
# ==========================================
# TODO: Replace with your actual keys
GOOGLE_API_KEY = "YOUR_GEMINI_API_KEY"
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN"

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
genai.configure(api_key=GOOGLE_API_KEY)

# Authenticate ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# ==========================================
# FLASK APP SETUP
# ==========================================
app = Flask(__name__)
CORS(app) # Allow all origins (Simple & Robust)

# GLOBAL MEMORY STORAGE
# Format: { 'user_1': [msg1, msg2], 'user_2': [] }
USER_SESSIONS = {}

# Initialize Models
sanitizer_model = genai.GenerativeModel('gemini-1.5-flash')
solver_model = genai.GenerativeModel('gemini-1.5-flash')
verifier_model = genai.GenerativeModel('gemini-1.5-flash')
tutor_model = genai.GenerativeModel('gemini-1.5-flash', generation_config={"response_mime_type": "application/json"})

# ==========================================
# PIPELINE LOGIC
# ==========================================
def safe_generate(model, prompt_parts):
    """Safely generate content, handling blocked/empty responses."""
    try:
        response = model.generate_content(prompt_parts)
        if response.candidates and response.candidates[0].content.parts:
            return response.text
        else:
            print(f"Warning: Model returned empty response. Finish Reason: {response.candidates[0].finish_reason}")
            return ""
    except Exception as e:
        print(f"Generation Error: {e}")
        return ""

def format_history(history):
    """Format history list into a readable string."""
    if not history:
        return "No previous history."
    
    formatted = ""
    for msg in history:
        role = msg.get('role', 'unknown').upper()
        content = msg.get('content', '')
        formatted += f"{role}: {content}\n"
    return formatted

def run_pipeline(image=None, user_text=None, history=None):
    pipeline_log = {}
    history = history or []
    history_text = format_history(history)

    # --- STAGE 1 & 2: Context Awareness & Extraction ---
    sanitizer_prompt = """
    Your goal is to identify the MAIN MATH PROBLEM being discussed.
    1. Look at the image (if provided).
    2. Look at the user's current message.
    3. Look at the conversation history.
    
    Extract ONLY the math problem itself (e.g., "Solve 2x+5=10" or "Calculate the area of...").
    If the user is just replying to a hint (e.g., "Is it 5?"), look back at the history to find the original problem.
    Ignore conversational filler.
    """
    
    prompt_parts = [sanitizer_prompt]
    
    if history:
        prompt_parts.append(f"\nConversation History:\n{history_text}\n")

    if image:
        prompt_parts.append("\n[User Uploaded Image]\n")
        prompt_parts.append(image)
    
    if user_text:
        prompt_parts.append(f"\nUser Current Message: {user_text}\n")

    clean_problem = safe_generate(sanitizer_model, prompt_parts).strip()
    pipeline_log['stage_1_2_context_extraction'] = clean_problem

    if not clean_problem:
        return {
            "explanation": "I'm a bit lost. Could you remind me what math problem we're working on?",
            "encouragement": "Just type the problem again!",
            "debug_pipeline": pipeline_log
        }

    # --- STAGE 3: Dual-Model Verification ---
    # 3a. Solver
    solver_prompt = f"Solve this math problem step-by-step: {clean_problem}"
    initial_solution = safe_generate(solver_model, solver_prompt)
    pipeline_log['stage_3a_solution'] = initial_solution

    # 3b. Verifier
    verifier_prompt = f"""
    Review this solution for correctness. 
    Problem: {clean_problem}
    Solution: {initial_solution}
    
    If it is correct, say 'CORRECT'. 
    If it is incorrect, provide the corrected solution.
    """
    verification = safe_generate(verifier_model, verifier_prompt)
    pipeline_log['stage_3b_verification'] = verification

    ground_truth = initial_solution if "CORRECT" in verification else verification

    # --- STAGE 4: The Socratic Tutor ---
    tutor_prompt = f"""
    You are a Socratic Math Tutor. 
    
    Current Problem: {clean_problem}
    Correct Solution (DO NOT REVEAL): {ground_truth}
    
    Conversation History:
    {history_text}
    
    User's Latest Input: {user_text if user_text else '[Image Uploaded]'}
    
    Your Goal: Help the student solve it themselves.
    1. Do NOT give the answer.
    2. If the user is wrong, guide them gently.
    3. If the user is right, congratulate them and ask if they want another problem.
    4. Be encouraging.
    
    Return JSON: {{ "explanation": "Your response here...", "encouragement": "..." }}
    """
    final_response = safe_generate(tutor_model, tutor_prompt)
    
    try:
        result = json.loads(final_response)
        result['debug_pipeline'] = pipeline_log
        return result
    except:
        return {
            "explanation": final_response,
            "encouragement": "Let's keep going!",
            "debug_pipeline": pipeline_log
        }

# ==========================================
# API ROUTES
# ==========================================
@app.route('/api/chat', methods=['POST'])
def chat():
    data = request.json
    user_message = data.get('message', '')
    user_id = data.get('userId', 'default_user') # Get User ID
    
    # Initialize history if not present
    if user_id not in USER_SESSIONS:
        USER_SESSIONS[user_id] = []
    
    # Add user message to history
    USER_SESSIONS[user_id].append({'role': 'user', 'content': user_message})
    
    try:
        response = run_pipeline(user_text=user_message, history=USER_SESSIONS[user_id])
        
        # Add assistant response to history
        assistant_msg = f"{response.get('explanation', '')} {response.get('encouragement', '')}"
        USER_SESSIONS[user_id].append({'role': 'assistant', 'content': assistant_msg})
        
        return jsonify(response)
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/analyze', methods=['POST'])
def analyze_image():
    if 'image' not in request.files:
        return jsonify({"error": "No image uploaded"}), 400
    file = request.files['image']
    user_message = request.form.get('message', '')
    user_id = request.form.get('userId', 'default_user') # Get User ID
    
    if user_id not in USER_SESSIONS:
        USER_SESSIONS[user_id] = []
        
    # Add user interaction to history
    USER_SESSIONS[user_id].append({'role': 'user', 'content': f"[Image Uploaded] {user_message}"})
    
    try:
        image = Image.open(file.stream)
        response = run_pipeline(image=image, user_text=user_message, history=USER_SESSIONS[user_id])
        
        # Add assistant response to history
        assistant_msg = f"{response.get('explanation', '')} {response.get('encouragement', '')}"
        USER_SESSIONS[user_id].append({'role': 'assistant', 'content': assistant_msg})
        
        return jsonify(response)
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/reset', methods=['POST'])
def reset_session():
    data = request.json
    user_id = data.get('userId', 'default_user')
    if user_id in USER_SESSIONS:
        USER_SESSIONS[user_id] = []
    return jsonify({"status": "success", "message": f"Memory cleared for {user_id}"})

# ==========================================
# RUN SERVER
# ==========================================
# Open a tunnel to port 5000
public_url = ngrok.connect(5000).public_url
print(f"\n🚀 Public URL: {public_url}\n")
print("Copy this URL and use it in your Frontend!\n")

if __name__ == '__main__':
    app.run(port=5000)